## Simulation


This notebook outlines a workflow for running a set of 2D Navier-Stokes (CFD) simulations, extracting results for machine learning and generating datasets for training surrogate models.   

The code below:
- sets up simulation parameters
- runs PyFR CFD cases across different configurations
- processes the data for downstream ML tasks

Next --> main-training.ipynb for building the surrogate model!

In [ ]:
import numpy as np
import pandas as pd
import itertools

from pathlib import Path

from process_results import process_sim_results
from extract_data import extract_all_cases
from run_sim import PyfrSimulation
from animate import vtu_to_mp4
from plots import plot_column

ROOT = Path().resolve().parent

SIM_NAME = "example-model2"

### Build Cases




In [ ]:
# Set parameter ranges
nu_rng = (0.01, 0.02)      # [nu, m/s^2] kintematic velocity, 0.01 for Re ~ 100 @ 1 Uin
uin_rng = (1.0, 2.0)       # [Uin, m/s] inlet velocity, free-stream velocity (nondim)
dt_rng = (0.05, 0.05)      # [dt, s] solver timestep (adjust if unstable)
tend_rng = (2.0, 20.0)     # [tend, s] total time (long enough for a few vortex shedding cycles)
dt_out_rng = (0.05, 0.10)  # [dt_out] save state interval
PERMS = 1                  # number of permutations per range

# Build cases
param_ranges = [
    np.linspace(*nu_rng, PERMS),
    np.linspace(*uin_rng, PERMS),
    np.linspace(*dt_rng, PERMS),
    np.linspace(*tend_rng, PERMS),
    np.linspace(*dt_out_rng, PERMS),
]

cases = [list(params) for params in itertools.product(*param_ranges)]
len(cases)

### Run CFD for Training Data

In [ ]:
%%time

# run simulation (Test meshes)
MESH_FILE = "../assets/config/2d-cylinder.msh"
PYFRM_FILE = "../assets/config/2d-cylinder.pyfrm"
INI_FILE = "../assets/config/2d-cylinder.ini"

m = PyfrSimulation(SIM_NAME, mesh_file=MESH_FILE, pyfrm_file=PYFRM_FILE)
m.run_bulk(cases, backend="metal", show_progress=True)

# Note, it can take several minutes per case.

### Process Results

In [ ]:
# Convert VTKs to PVD files for easier post-processing
process_sim_results(SIM_NAME)

In [ ]:
# Extracts and exports nodewise results for a single simulation case. This is 
# then used for training our GNN.
df = extract_all_cases(SIM_NAME).to_pandas()  # plots use pandas
df.head()

### Visualise Raw Results

In [ ]:
# Generate MP4 for simulation results (optional)
FPS = 20                     # Frames per second (match 1/dt)
CMAP = "turbo"               # Color map
OFF_SCREEN = True            # Render offscreen
WINDOW_SIZE = (1920, 1088)   # Resolution of rendered frames (width, height) 

for i in range(len(cases)):
    print(f"Generating animation for Case: 'case{i}'")
    vtu_to_mp4(
        SIM_NAME, 
        f"case{i}", 
        remove_images=True,
        fps=FPS,
        cmap=CMAP,
        off_screen=OFF_SCREEN,
        window_size=WINDOW_SIZE
    )

# MP4s saved in 'sims/<sim_name>/animations/<sim_name>_<case_idx>.mp4'

In [ ]:
# Visualise static results
CASE = "case0" 
STEP = 15
METRIC = "p" # p (pressure), u (x-velocity), v (y-velocity), vn (normal velocity)
    
if 'df' not in locals() and 'df' not in globals():
    results_path = ROOT / "sims" / SIM_NAME / "ml_training" / f"{CASE}-results.csv"
    df = pd.read_csv(results_path)

plot_column(df, METRIC, step=STEP)